In [1]:
import csv
import json
import math
import re
import statistics
from pathlib import Path

In [2]:



# ============================================================
# SETTINGS
# ============================================================

folder = Path.cwd()
data_folder = folder / "csv_data"
output_file = folder / "index.html"

RMS_LIMIT_PN = 25.0
RMS_COLUMN = "elasticity_rms_bruker_pN"
KB_COLUMN = "bacterial_spring_constant_nN_per_um"
E_COLUMN = "youngs_modulus_MPa"

CONDITION_ORDER = [
    "Live",
    "20 mg/L",
    "320 mg/L",
    "Dead",
]

FOLDER_PATTERN = re.compile(
    r"^(?P<condition>Live|20|320|Dead)_cell(?P<cell>\d+)_crop(?P<crop>\d+)$",
    re.IGNORECASE,
)

CONDITION_LABELS = {
    "live": "Live",
    "20": "20 mg/L",
    "320": "320 mg/L",
    "dead": "Dead",
}


# ============================================================
# HELPERS
# ============================================================

def read_csv(path):
    with path.open(encoding="utf-8-sig", newline="") as file:
        return list(csv.DictReader(file))


def get_points(rows, column, used_column=None):
    points = []

    for row in rows:
        if used_column and row.get(used_column) != "1":
            continue

        try:
            x = float(row["x_nm"])
            y = float(row[column])
        except (KeyError, TypeError, ValueError):
            continue

        if math.isfinite(x) and math.isfinite(y):
            points.append([x, y])

    if used_column:
        points.sort(key=lambda point: point[0])

    return points


def curve_key(row):
    return (
        row.get("filename", ""),
        row.get("curve_number", ""),
        row.get("position_index", ""),
    )


def safe_float(value):
    try:
        number = float(value)
    except (TypeError, ValueError):
        return None

    return number if math.isfinite(number) else None


def median_absolute_deviation(values):
    clean = [
        float(value)
        for value in values
        if value is not None and math.isfinite(float(value))
    ]

    if not clean:
        return None

    median_value = statistics.median(clean)
    deviations = [abs(value - median_value) for value in clean]
    return statistics.median(deviations)


def format_stat(value, digits=4):
    if value is None:
        return ""

    try:
        number = float(value)
    except (TypeError, ValueError):
        return ""

    if not math.isfinite(number):
        return ""

    return f"{number:.{digits}f}"


# ============================================================
# LOAD CURVES FOR THE INTERACTIVE DASHBOARD
# ============================================================

datasets = []

curve_folders = sorted({
    path.parent
    for path in data_folder.rglob("AFM_fit_points_curve_*.csv")
})

for curve_folder in curve_folders:
    summary_file = curve_folder / "AFM_jython_summary.csv"

    if not summary_file.exists():
        raise FileNotFoundError(
            f"Missing summary CSV in {curve_folder}"
        )

    summary = {
        curve_key(row): row
        for row in read_csv(summary_file)
    }

    curves = []

    for path in sorted(
        curve_folder.glob("AFM_fit_points_curve_*.csv")
    ):
        rows = read_csv(path)

        if not rows:
            print("Skipped empty file:", path.name)
            continue

        first = rows[0]
        values = summary.get(curve_key(first), {})

        if not values:
            print("No matching summary row:", path.name)

        curves.append({
            "number": first["curve_number"],
            "position": first["position_index"],
            "filename": first["filename"],
            "measured": get_points(
                rows,
                "measured_y_nN",
            ),
            "linear": get_points(
                rows,
                "linear_fit_y_nN",
                "linear_fit_used",
            ),
            "elasticity": get_points(
                rows,
                "elasticity_fit_y_nN",
                "elasticity_fit_used",
            ),
            "values": values,
        })

    curves.sort(
        key=lambda curve: (
            int(curve["number"]),
            int(curve["position"]),
        )
    )

    if curves:
        datasets.append({
            "name": curve_folder.relative_to(data_folder).as_posix(),
            "curves": curves,
        })

if not datasets:
    raise SystemExit(
        "No curves found. Put your CSV files inside csv_data."
    )


# ============================================================
# BUILD THE SUMMARY TABLE DATA
# ============================================================
#
# Crop folders for the same condition and cell are pooled.
# Example:
#   Live_cell1_crop1
#   Live_cell1_crop2
# become:
#   Live, Cell 1
#
# Inclusion criterion:
#   JPK elasticity RMS <= 25 pN
#
# Per cell:
#   n, median kB, MAD kB, median E, MAD E
#
# Per condition:
#   mean of cell medians and sample SD of cell medians
#
# ============================================================

summary_groups = {}

for curve_folder in curve_folders:
    match = FOLDER_PATTERN.match(curve_folder.name)

    if match is None:
        continue

    condition = CONDITION_LABELS[
        match.group("condition").lower()
    ]
    cell_number = int(match.group("cell"))
    summary_file = curve_folder / "AFM_jython_summary.csv"

    if not summary_file.exists():
        continue

    group_key = (condition, cell_number)
    summary_groups.setdefault(group_key, [])

    for row in read_csv(summary_file):
        rms = safe_float(row.get(RMS_COLUMN))
        kb = safe_float(row.get(KB_COLUMN))
        youngs = safe_float(row.get(E_COLUMN))

        if rms is None or rms > RMS_LIMIT_PN:
            continue

        if kb is None and youngs is None:
            continue

        summary_groups[group_key].append({
            "rms": rms,
            "kb": kb,
            "E": youngs,
            "curve_number": row.get("curve_number", ""),
            "position_index": row.get("position_index", ""),
            "source_folder": curve_folder.name,
        })


cell_stats = {}

for key, rows in summary_groups.items():
    kb_values = [
        row["kb"]
        for row in rows
        if row["kb"] is not None
    ]

    e_values = [
        row["E"]
        for row in rows
        if row["E"] is not None
    ]

    cell_stats[key] = {
        "n": len(rows),
        "kb_median": (
            statistics.median(kb_values)
            if kb_values
            else None
        ),
        "kb_mad": median_absolute_deviation(kb_values),
        "e_median": (
            statistics.median(e_values)
            if e_values
            else None
        ),
        "e_mad": median_absolute_deviation(e_values),
    }


condition_stats = {}

for condition in CONDITION_ORDER:
    cells = sorted(
        cell_number
        for group_condition, cell_number in cell_stats
        if group_condition == condition
    )

    kb_medians = [
        cell_stats[(condition, cell)]["kb_median"]
        for cell in cells
        if cell_stats[(condition, cell)]["kb_median"] is not None
    ]

    e_medians = [
        cell_stats[(condition, cell)]["e_median"]
        for cell in cells
        if cell_stats[(condition, cell)]["e_median"] is not None
    ]

    condition_stats[condition] = {
        "cells": cells,
        "kb_mean_of_medians": (
            statistics.mean(kb_medians)
            if kb_medians
            else None
        ),
        "kb_sd": (
            statistics.stdev(kb_medians)
            if len(kb_medians) >= 2
            else None
        ),
        "e_mean_of_medians": (
            statistics.mean(e_medians)
            if e_medians
            else None
        ),
        "e_sd": (
            statistics.stdev(e_medians)
            if len(e_medians) >= 2
            else None
        ),
    }


summary_rows_html = []

for condition in CONDITION_ORDER:
    group = condition_stats[condition]
    cells = group["cells"]

    if not cells:
        continue

    rowspan = len(cells)

    for index, cell_number in enumerate(cells):
        stats = cell_stats[(condition, cell_number)]

        condition_cell = ""
        kb_group_cells = ""
        e_group_cells = ""

        if index == 0:
            condition_cell = (
                f'<td class="summary-condition" rowspan="{rowspan}">'
                f'{condition}</td>'
            )

            kb_group_cells = (
                f'<td class="summary-group-value" rowspan="{rowspan}">'
                f'{format_stat(group["kb_mean_of_medians"], 2)}</td>'
                f'<td class="summary-group-value" rowspan="{rowspan}">'
                f'{format_stat(group["kb_sd"], 2)}</td>'
            )

            e_group_cells = (
                f'<td class="summary-group-value" rowspan="{rowspan}">'
                f'{format_stat(group["e_mean_of_medians"], 2)}</td>'
                f'<td class="summary-group-value" rowspan="{rowspan}">'
                f'{format_stat(group["e_sd"], 2)}</td>'
            )

        summary_rows_html.append(
            "<tr>"
            + condition_cell
            + f"<td>Cell {cell_number}</td>"
            + f'<td>{stats["n"]}</td>'
            + f'<td>{format_stat(stats["kb_median"], 4)}</td>'
            + f'<td>{format_stat(stats["kb_mad"], 4)}</td>'
            + kb_group_cells
            + f'<td>{format_stat(stats["e_median"], 4)}</td>'
            + f'<td>{format_stat(stats["e_mad"], 4)}</td>'
            + e_group_cells
            + "</tr>"
        )


if summary_rows_html:
    summary_table_html = f"""
    <div class="card summary-card">
        <h2>AFM summary, JPK elasticity RMS ≤ {RMS_LIMIT_PN:g} pN</h2>

        <div class="summary-table-wrapper">
            <table class="summary-table">
                <thead>
                    <tr>
                        <th rowspan="2">Condition</th>
                        <th rowspan="2">Cell</th>
                        <th>RMS ≤ {RMS_LIMIT_PN:g} pN</th>
                        <th colspan="4">kB (nN/µm)</th>
                        <th colspan="4">E (MPa)</th>
                    </tr>
                    <tr>
                        <th>n</th>
                        <th>Median</th>
                        <th>MAD</th>
                        <th>Mean of medians</th>
                        <th>SD</th>
                        <th>Median</th>
                        <th>MAD</th>
                        <th>Mean of medians</th>
                        <th>SD</th>
                    </tr>
                </thead>
                <tbody>
                    {''.join(summary_rows_html)}
                </tbody>
            </table>
        </div>
    </div>
    """
else:
    summary_table_html = f"""
    <div class="card summary-card">
        <h2>AFM summary, JPK elasticity RMS ≤ {RMS_LIMIT_PN:g} pN</h2>
        <p>No matching condition/cell crop folders were found.</p>
    </div>
    """


# ============================================================
# HTML PAGE
# ============================================================

html = r"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <title>AFM CSV Dashboard</title>

    <script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>

    <style>
        * {
            box-sizing: border-box;
        }

        body {
            margin: 0;
            font-family: Arial, sans-serif;
            background: #f4f5f7;
            color: #202124;
        }

        header {
            padding: 20px 28px;
            background: white;
            border-bottom: 1px solid #ddd;
        }

        h1 {
            margin: 0;
            font-size: 24px;
        }

        header p {
            margin-bottom: 0;
            color: #666;
        }

        .controls,
        .card {
            background: white;
            padding: 20px;
            border-radius: 8px;
        }

        .controls {
            margin: 20px;
            display: flex;
            align-items: center;
            flex-wrap: wrap;
            gap: 12px;
        }

        select,
        button {
            padding: 9px 12px;
            font-size: 14px;
        }

        select {
            min-width: 200px;
            max-width: 100%;
        }

        button {
            cursor: pointer;
        }

        .layout {
            display: grid;
            grid-template-columns: minmax(0, 1fr) 360px;
            gap: 20px;
            margin: 20px;
        }

        .card {
            min-width: 0;
        }

        #plot {
            height: 600px;
        }

        h2 {
            margin-top: 0;
            font-size: 20px;
        }

        table {
            width: 100%;
            border-collapse: collapse;
            font-size: 14px;
        }

        #values td {
            padding: 10px 4px;
            border-bottom: 1px solid #eee;
        }

        #values td:nth-child(2) {
            text-align: right;
            font-weight: bold;
        }

        #values td:last-child {
            color: #666;
            padding-left: 10px;
            white-space: nowrap;
        }

        #filename,
        #status {
            font-size: 13px;
            color: #666;
            line-height: 1.5;
            overflow-wrap: anywhere;
        }

        .summary-card {
            margin: 20px;
        }

        .summary-table-wrapper {
            overflow-x: auto;
        }

        .summary-table {
            width: 100%;
            border-collapse: collapse;
            font-size: 14px;
        }

        .summary-table th,
        .summary-table td {
            padding: 9px 10px;
            border: 1px solid #d9dde2;
            text-align: center;
            vertical-align: middle;
            white-space: nowrap;
        }

        .summary-table th {
            background: #f5f6f7;
            font-weight: 700;
        }

        .summary-table .summary-condition {
            font-weight: 700;
            background: #fafafa;
        }

        .summary-table .summary-group-value {
            font-weight: 600;
        }

        @media (max-width: 900px) {
            .layout {
                grid-template-columns: 1fr;
            }

            #plot {
                height: 450px;
            }
        }
    </style>
</head>

<body>
    <header>
        <h1>AFM CSV Dashboard</h1>
        <p>Force curves and saved fit results</p>
    </header>

    <div class="controls">
        <label for="folder">Folder</label>
        <select id="folder"></select>

        <label for="curve">Curve</label>
        <select id="curve"></select>

        <button id="previous">Previous</button>
        <button id="next">Next</button>
    </div>

    <div class="layout">
        <div class="card">
            <div id="plot"></div>
        </div>

        <div class="card">
            <h2>Curve values</h2>

            <table>
                <tbody id="values"></tbody>
            </table>

            <p id="status"></p>
            <p id="filename"></p>
        </div>
    </div>

    __SUMMARY_TABLE__

    <script>
        const datasets = __DATA__;

        const folderSelect = document.getElementById("folder");
        const curveSelect = document.getElementById("curve");
        const valuesTable = document.getElementById("values");
        const status = document.getElementById("status");
        const filenameDisplay = document.getElementById("filename");

        const fields = [
            ["Linear slope", "linear_slope_bruker_N_per_m", "N/m"],
            ["Bacterial spring constant", "bacterial_spring_constant_N_per_m", "N/m"],
            ["Bacterial spring constant", "bacterial_spring_constant_nN_per_um", "nN/µm"],
            ["Linear RMSD custom", "linear_rmsd_custom_pN", "pN"],
            ["Linear X minimum", "linear_x_min_used_nm", "nm"],
            ["Linear X maximum", "linear_x_max_used_nm", "nm"],
            ["Linear fit points", "linear_T", ""],
            ["Young's modulus", "youngs_modulus_MPa", "MPa"],
            ["JPK Elasticity RMS", "elasticity_rms_bruker_pN", "pN"],
            ["Elasticity RMSD custom", "elasticity_rmsd_custom_pN", "pN"],
            ["Elasticity X minimum", "elasticity_x_min_used_nm", "nm"],
            ["Elasticity X maximum", "elasticity_x_max_used_nm", "nm"],
            ["Elasticity fit points", "elasticity_T", ""],
            ["Elasticity fit points till contact", "elasticity_contact_T", ""],
        ];

        function formatNumber(value) {
            if (value == null || String(value).trim() === "") {
                return "N/A";
            }

            const number = Number(value);

            return Number.isFinite(number)
                ? String(Number(number.toPrecision(6)))
                : "N/A";
        }

        function percentageDifference(value1, value2) {
            const a = Number(value1);
            const b = Number(value2);

            if (!Number.isFinite(a) || !Number.isFinite(b)) {
                return "N/A";
            }

            const mean = (Math.abs(a) + Math.abs(b)) / 2;

            if (mean === 0) {
                return "0";
            }

            const difference = Math.abs(a - b) / mean * 100;

            return String(Number(difference.toPrecision(6)));
        }

        function appendValueRow(label, value, unit) {
            const row = document.createElement("tr");

            for (const text of [label, value, unit]) {
                const cell = document.createElement("td");
                cell.textContent = text;
                row.appendChild(cell);
            }

            valuesTable.appendChild(row);
        }

        function makeTrace(points, name, color, mode) {
            return {
                x: points.map(point => point[0]),
                y: points.map(point => point[1]),
                type: "scatter",
                mode: mode,
                name: name,
                marker: { color: color, size: 3 },
                line: { color: color, width: 3 }
            };
        }

        function showCurve() {
            const dataset = datasets[Number(folderSelect.value)];
            const index = Number(curveSelect.value);
            const curve = dataset.curves[index];

            const traces = [
                {
                    x: curve.measured.map(point => point[0]),
                    y: curve.measured.map(point => point[1]),
                    type: "scatter",
                    mode: "lines+markers",
                    name: "Measured",
                    marker: {
                        color: "#58758a",
                        size: 4
                    },
                    line: {
                        color: "#58758a",
                        width: 1.5
                    }
                }
            ];

            if (curve.linear.length) {
                traces.push(
                    makeTrace(
                        curve.linear,
                        "Linear fit",
                        "#2ca02c",
                        "lines"
                    )
                );
            }

            if (curve.elasticity.length) {
                traces.push(
                    makeTrace(
                        curve.elasticity,
                        "Elasticity fit",
                        "#d62728",
                        "lines"
                    )
                );
            }

            Plotly.react(
                "plot",
                traces,
                {
                    title: `${dataset.name} | Curve ${curve.number}`,
                    xaxis: { title: "X (nm)", autorange: true },
                    yaxis: { title: "Force (nN)", autorange: true },
                    legend: { orientation: "h", y: -0.2 },
                    margin: { l: 70, r: 25, t: 60, b: 90 },
                    hovermode: "closest"
                },
                {
                    responsive: true,
                    displaylogo: false,
                    toImageButtonOptions: {
                        filename: `AFM_curve_${curve.number}`,
                        scale: 2
                    }
                }
            );

            valuesTable.replaceChildren();

            for (const [label, column, unit] of fields) {
                appendValueRow(
                    label,
                    formatNumber(curve.values[column]),
                    unit
                );

                if (column === "elasticity_rmsd_custom_pN") {
                    const rmsDifference = percentageDifference(
                        curve.values["elasticity_rms_bruker_pN"],
                        curve.values["elasticity_rmsd_custom_pN"]
                    );

                    appendValueRow(
                        "Elasticity RMS percentage difference",
                        rmsDifference,
                        "%"
                    );
                }
            }

            status.textContent =
                `Curve ${index + 1} of ${dataset.curves.length}. ` +
                `Position index: ${curve.position}.`;

            if (Object.keys(curve.values).length === 0) {
                status.textContent += " No matching summary row.";
            }

            filenameDisplay.textContent = curve.filename;

            document.getElementById("previous").disabled = index === 0;
            document.getElementById("next").disabled =
                index === dataset.curves.length - 1;
        }

        function showFolder() {
            const dataset = datasets[Number(folderSelect.value)];
            curveSelect.replaceChildren();

            dataset.curves.forEach((curve, index) => {
                curveSelect.add(
                    new Option(
                        `Curve ${curve.number} | Position ${curve.position}`,
                        index
                    )
                );
            });

            showCurve();
        }

        datasets.forEach((dataset, index) => {
            folderSelect.add(new Option(dataset.name, index));
        });

        folderSelect.onchange = showFolder;
        curveSelect.onchange = showCurve;

        document.getElementById("previous").onclick = () => {
            if (curveSelect.selectedIndex > 0) {
                curveSelect.selectedIndex -= 1;
                showCurve();
            }
        };

        document.getElementById("next").onclick = () => {
            if (curveSelect.selectedIndex < curveSelect.length - 1) {
                curveSelect.selectedIndex += 1;
                showCurve();
            }
        };

        if (window.Plotly) {
            showFolder();
        } else {
            status.textContent =
                "Could not load the plotting library. Check your connection and reload.";
        }
    </script>
</body>
</html>
"""


# ============================================================
# EMBED DATA AND WRITE index.html
# ============================================================

data_json = json.dumps(
    datasets,
    ensure_ascii=True,
    allow_nan=False,
).replace("<", "\\u003c")

final_html = (
    html
    .replace("__DATA__", data_json)
    .replace("__SUMMARY_TABLE__", summary_table_html)
)

output_file.write_text(
    final_html,
    encoding="utf-8",
)


# ============================================================
# CONSOLE SUMMARY
# ============================================================

print("Created:", output_file.resolve())
print("Folders:", len(datasets))
print("Curves:", sum(len(item["curves"]) for item in datasets))
print(
    "RMS summary filter:",
    f"{RMS_COLUMN} <= {RMS_LIMIT_PN} pN"
)
print("Summary groups:")

for key in sorted(cell_stats):
    print(
        key,
        "n =",
        cell_stats[key]["n"],
    )


Created: /workspaces/AFM-plots_Jython/index.html
Folders: 6
Curves: 73
RMS summary filter: elasticity_rms_bruker_pN <= 25.0 pN
Summary groups:
('20 mg/L', 1) n = 33
('320 mg/L', 1) n = 13
('Live', 1) n = 27
